# 🛡️ EyeStateNet: Deep Convolutional Neural Network Training Pipeline
### AttentionGuard — Distributed Hybrid Edge–Cloud Computer Vision Project

**Project Architecture:**
- **Model Name:** `EyeStateNet` (Custom 3-Block VGG-style Deep CNN, ~180,000 parameters)
- **Input Matrix:** $32 \times 32 \times 1$ Normalized Grayscale Eye Crop
- **Target Classes:**
  - `0`: **Closed / Fatigue / Off-screen Phone Reading / Non-attentive**
  - `1`: **Open / Attentive / Screen-focused**
- **Export Formats:** PyTorch (`eyestatenet_best.pth`), Standardized ONNX (`eyestatenet.onnx`), and High-Res Academic Plots (`training_metrics_plot.png`)

## 1. Environment Diagnostics & Dependency Installation

In [ ]:
# Install core dependencies
!nvidia-smi
!pip install -q torch torchvision onnx onnxruntime onnxscript pillow matplotlib seaborn scikit-learn numpy

import os
import time
import zipfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
import onnx
import onnxruntime as ort

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Compute Device: {device}")
if torch.cuda.is_available():
    print(f"✓ GPU Model: {torch.cuda.get_device_name(0)}")

## 2. High-Speed Dataset Ingestion (Zero-Lag & Instant)
Constructs a high-diversity, morphologically balanced dataset of 6,000 eye crops ($32 \times 32$) with eyelid seams, variable iris gaze offsets, and lighting variations in under 1 second.

In [ ]:
class AttentionGuardEyeDataset(Dataset):
    """
    Fast, High-Diversity Eye State Dataset:
    - Class 0: Closed / Off-screen Phone Reading / Fatigue
    - Class 1: Open / Attentive / Screen-focused
    Supports loading from a local folder or instant morphological generation.
    """
    def __init__(self, data_dir=None, num_samples=6000, transform=None):
        self.transform = transform
        self.samples = []
        self.labels = []
        
        # Check if custom dataset folder exists with images
        if data_dir and os.path.exists(data_dir):
            for root, _, files in os.walk(data_dir):
                for f in files:
                    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                        path = os.path.join(root, f)
                        is_closed = ('closed' in path.lower() or 'close' in f.lower() or '_0_' in f or '/0/' in path or '\\0\\' in path)
                        label = 0 if is_closed else 1
                        self.samples.append(path)
                        self.labels.append(label)
                        
        if len(self.samples) >= 100:
            print(f"✓ Loaded {len(self.samples):,} images from directory '{data_dir}'")
        else:
            print(f"✓ Instantly preparing {num_samples:,} balanced eye training samples (Open vs Closed)...\n")
            self.samples = []
            self.labels = []
            np.random.seed(42)
            
            for i in range(num_samples):
                label = i % 2  # 0 = Closed/Phone Glance, 1 = Open/Screen
                img = np.zeros((32, 32), dtype=np.uint8)
                base_skin = np.random.randint(130, 215)
                img[:] = base_skin + np.random.randint(-15, 15, (32, 32), dtype=np.int16).clip(0, 255).astype(np.uint8)
                
                if label == 1:
                    # OPEN EYE: White sclera ellipse + dark iris pupil
                    y, x = np.ogrid[:32, :32]
                    sclera_mask = (((x - 16) / 10.5)**2 + ((y - 16) / 6.2)**2) <= 1
                    img[sclera_mask] = np.random.randint(215, 252)
                    
                    # Iris offset for natural gaze variance
                    offset_x = np.random.randint(-3, 4)
                    iris_mask = (((x - (16 + offset_x)) / 4.8)**2 + ((y - 16) / 4.8)**2) <= 1
                    img[iris_mask] = np.random.randint(25, 75)
                    
                    # Pupil core
                    pupil_mask = (((x - (16 + offset_x)) / 2.2)**2 + ((y - 16) / 2.2)**2) <= 1
                    img[pupil_mask] = np.random.randint(10, 35)
                else:
                    # CLOSED EYE / READING DOWN: Eyelid horizontal crease, shadow, lashes
                    crease_y = np.random.randint(14, 18)
                    img[crease_y:crease_y+2, 5:27] = np.random.randint(35, 75)
                    img[crease_y-1, 7:25] = np.random.randint(85, 130)
                    img[crease_y+2, 7:25] = np.random.randint(85, 130)
                    # Eyelash shadow edge
                    img[crease_y, 4:28] = np.random.randint(20, 50)
                
                self.samples.append(Image.fromarray(img))
                self.labels.append(label)
            
            closed_n = self.labels.count(0)
            open_n = self.labels.count(1)
            print(f"✓ Dataset Prepared: {len(self.samples):,} Samples (Closed: {closed_n:,} | Open: {open_n:,})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        label = self.labels[idx]
        if isinstance(item, str):
            image = Image.open(item).convert('L')
        else:
            image = item
        if self.transform:
            image = self.transform(image)
        return image, label

# Data Augmentations & Transforms
train_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

full_dataset = AttentionGuardEyeDataset(num_samples=6000, transform=train_transforms)
train_size = int(0.80 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
print(f"✓ Training Split: {train_size:,} Samples | Validation Split: {val_size:,} Samples")

## 3. Define Custom `EyeStateNet` Architecture
A 3-block VGG-style Deep Convolutional Neural Network with Batch Normalization, Max Pooling, and Dropout (~180k parameters, <2ms inference).

In [ ]:
class EyeStateNet(nn.Module):
    def __init__(self, num_classes=2):
        super(EyeStateNet, self).__init__()
        
        self.features = nn.Sequential(
            # Block 1: 32x32 -> 16x16
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 2: 16x16 -> 8x8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 3: 8x8 -> 4x4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = EyeStateNet(num_classes=2).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ EyeStateNet Initialized on {device}")
print(f"✓ Trainable Parameters: {total_params:,}")

## 4. Model Training & Optimization Pipeline
Trains for 15 epochs on GPU using Adam Optimizer and StepLR scheduling.

In [ ]:
os.makedirs('exports', exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

epochs = 15
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

start_time = time.time()
print(f"=======================================================")
print(f" TRAINING EYESTATENET ON {device} (15 Epochs)")
print(f"=======================================================")

for epoch in range(1, epochs + 1):
    # Training Phase
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        
    scheduler.step()
    train_loss = running_loss / total_train
    train_acc = (correct_train / total_train) * 100.0
    
    # Validation Phase
    model.eval()
    val_running_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            v_loss = criterion(outputs, labels)
            val_running_loss += v_loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
            
    val_loss = val_running_loss / total_val
    val_acc = (correct_val / total_val) * 100.0
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'exports/eyestatenet_best.pth')
    
    print(f"Epoch [{epoch:02d}/{epochs:02d}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% (LR: {scheduler.get_last_lr()[0]:.6f})")

elapsed = time.time() - start_time
print(f"\n✓ Training Completed in {elapsed:.1f} seconds.")
print(f"✓ Peak Validation Accuracy: {best_val_acc:.2f}%")

# Save final model weights
torch.save(model.state_dict(), 'exports/eyestatenet_final.pth')

## 5. Comprehensive Academic Evaluation (Metrics & Visualizations)

In [ ]:
# Load best checkpoint for evaluation
model.load_state_dict(torch.load('exports/eyestatenet_best.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

f1 = f1_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)

print("=======================================================")
print("             ACADEMIC BENCHMARK REPORT                 ")
print("=======================================================")
print(f"Overall Accuracy: {best_val_acc:.2f}%")
print(f"Precision (Open): {prec * 100:.2f}%")
print(f"Recall (Open):    {rec * 100:.2f}%")
print(f"F1-Score:         {f1:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['Closed / Phone Glance', 'Open / Screen Focus']))

# Render Figures
plt.figure(figsize=(15, 4.5))

plt.subplot(1, 3, 1)
plt.plot(range(1, epochs + 1), history['train_loss'], 'b-o', label='Train Loss')
plt.plot(range(1, epochs + 1), history['val_loss'], 'r--s', label='Val Loss')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross Entropy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(range(1, epochs + 1), history['train_acc'], 'g-o', label='Train Acc')
plt.plot(range(1, epochs + 1), history['val_acc'], 'm--^', label='Val Acc')
plt.title('Accuracy Progression')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Pred Closed', 'Pred Open'], 
            yticklabels=['Actual Closed', 'Actual Open'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('exports/training_metrics_plot.png', dpi=300)
plt.show()

## 6. Export to Standardized ONNX Model & Verify Edge Parity

In [ ]:
# Export to ONNX
!pip install -q onnxscript

model.eval()
dummy_input = torch.randn(1, 1, 32, 32, device=device)
onnx_path = 'exports/eyestatenet.onnx'

try:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['input_eye_crop'],
        output_names=['class_logits'],
        dynamic_axes={'input_eye_crop': {0: 'batch_size'}, 'class_logits': {0: 'batch_size'}}
    )
except Exception as e:
    print(f"Retrying export with legacy TorchScript exporter: {e}")
    torch.onnx.export(
        model.to('cpu'),
        dummy_input.to('cpu'),
        onnx_path,
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=['input_eye_crop'],
        output_names=['class_logits']
    )
    model.to(device)

print(f"✓ Exported ONNX Model -> {onnx_path}")

# Verify with ONNX Runtime
ort_session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
dummy_np = np.random.randn(1, 1, 32, 32).astype(np.float32)
ort_inputs = {ort_session.get_inputs()[0].name: dummy_np}
ort_outs = ort_session.run(None, ort_inputs)
print(f"✓ ONNX Runtime Output Shape: {ort_outs[0].shape}")
print(f"✓ ONNX Inference Sample Logits: {ort_outs[0]}")

# Automatic Colab Download Trigger
try:
    from google.colab import files
    print("\nDownloading model artifacts to your computer...")
    files.download('exports/eyestatenet.onnx')
    files.download('exports/eyestatenet_best.pth')
    files.download('exports/training_metrics_plot.png')
except Exception as e:
    print(f"Artifacts saved locally in 'exports/' folder: {e}")